# 01 – Preprocesamiento y modelado jerárquico de la taxonomía PAE

Este notebook implementa el proceso de transformación estructural de la taxonomía PAE, convirtiendo el fichero original en una representación jerárquica de dos niveles mediante la generación del campo parent_code.
En este esquema:
- Las categorías principales (1A0, 1B0, etc) constituyen el nivel raíz. Las categorías raíz no poseen padre (parent_code = None).
- El resto de categorías se asignan como hijas directas de su correspondiente categoría principal. Por ejemplo, las categorías 1A3 o 1A11 tendrán como parent_code a 1A0.


El resultado es un dataset estructurado, validado y listo para su utilización en experimentos de clasificación automática de proyectos de investigación dentro del dominio aeroespacial.

Este preprocesamiento constituye la primera fase del pipeline metodológico del proyecto, garantizando consistencia estructural y trazabilidad jerárquica antes de aplicar modelos de aprendizaje profundo o técnicas basadas en embeddings semánticos.

Esta celda importa las librerías necesarias para el procesamiento del fichero Excel.
Se utiliza:

- pandas para la manipulación estructurada de datos.
- re para el uso de expresiones regulares en la extracción y validación de los códigos de categoría.

In [26]:
from google.colab import files
# sube el fichero aerotaxLimpio_v3.xlsx
uploaded = files.upload()

Saving aerotaxLimpio_v3.xlsx to aerotaxLimpio_v3 (5).xlsx


In [27]:
import re
import pandas as pd


Ejecuta y sube el fichero de la taxonomía:

## Configuración y funciones auxiliares

En esta celda configuramos los elementos básicos necesarios para transformar la taxonomía original.

Primero definimos:

- El fichero de entrada (aerotaxLimpio_v3.xlsx).
- El fichero de salida (aerotax_transformado.csv).

La lista de códigos principales (ROOT_CODES), que identifican las categorías raíz.

Después configuramos las expresiones regulares que nos permiten:

- Extraer correctamente el código estructural de cada categoría.
- Separar el identificador del texto descriptivo.


Finalmente, definimos dos funciones:

- split_label(): divide el campo label en category_code y category_label, asegurando que el código tenga el formato correcto.
- compute_parent_code(): asigna el parent_code correspondiente. Si la categoría es principal → parent_code = None. Si no lo es → se asigna como padre la categoría raíz correspondiente (por ejemplo, 1A11 → 1A0).

Esta celda prepara toda la lógica necesaria para generar la estructura final del dataset transformado.

In [28]:
INPUT_XLSX = "aerotaxLimpio_v3.xlsx"
OUTPUT_FILE = "aerotax_transformado.csv"

ROOT_CODES = {
    '1A0', '1B0', '1C0', '1D0', '1E0', '1F0',
    '1G0', '1H0', '1I0', '1J0', '1K0', '1L0'
}

# Captura: prefijo (ej 1A) + número (ej 11)
CODE_RE = re.compile(r"^(\d+[A-Z])(\d+)$")
# Para partir label robustamente: '<CODE> <TEXTO...>'
LABEL_RE = re.compile(r"^(\d+[A-Z]\d+)\b\s*(.*)$")

def split_label(label: str):
    """
    label: '<CODE> <TEXTO...>'
    Ej: '1A11 External Noise Prediction'
    """
    s = str(label).strip()
    m = LABEL_RE.match(s)
    if not m:
        raise ValueError(f"Label mal formado (sin código válido al inicio): {s}")
    return m.group(1), m.group(2).strip()


def compute_parent_code(code: str):
    """
    parent_code:
    - None si es root (1A0..1L0)
    - si num tiene 1 dígito -> prefijo + '0' (ej 1A7 -> 1A0, 1A1 -> 1A0)
    - si num tiene >1 dígito -> prefijo + num sin último dígito (ej 1A11 -> 1A1)
    """
    if code in ROOT_CODES:
        return None

    m = CODE_RE.match(str(code).strip())
    if not m:
        return None  # o lanzar error si prefieres

    prefix, num = m.group(1), m.group(2)

    return prefix + "0"


def clean_text(text):
    if pd.isna(text):
        return text

    text = str(text)

    # Eliminar marcador Excel
    text = text.replace("_x000D_", " ")

    # Eliminar caracteres de control Unicode
    text = re.sub(r"[\r\n\t]", " ", text)

    # Normalizar espacios
    text = re.sub(r"\s+", " ", text)

    return text.strip()




## Carga y transformación de la taxonomía

En esta celda se realiza la transformación efectiva del fichero original.

Primero se carga el archivo Excel definido en la configuración. A continuación, se verifica que el dataset contenga las columnas necesarias (label y text). Esta validación evita errores posteriores si la estructura del fichero no es la esperada.

Después:

- Se aplica la función split_label() sobre la columna label para extraer:
category_code (identificador estructural de la categoría) y category_label (nombre textual de la categoría).
- Se calcula el parent_code para cada categoría utilizando la función compute_parent_code(), lo que permite explicitar la relación estructural con su categoría principal.

Finalmente, se construye el DataFrame df_out, que contiene únicamente las columnas relevantes para el uso posterior en el pipeline:

- category_code
- category_label
- parent_code
- text

La llamada a head() permite inspeccionar visualmente las primeras filas y verificar que la transformación se ha realizado correctamente.

In [29]:
df = pd.read_excel(INPUT_XLSX)

if not {"label", "text"}.issubset(df.columns):
    raise ValueError(f"Se esperaban columnas 'label' y 'text'. Encontradas: {list(df.columns)}")

codes_labels = df["label"].apply(split_label)
df["category_code"] = codes_labels.apply(lambda x: x[0])
df["category_label"] = codes_labels.apply(lambda x: x[1])

df["parent_code"] = df["category_code"].apply(compute_parent_code)

df["text"] = df["text"].apply(clean_text)

df_out = df[["category_code", "category_label", "parent_code", "text"]].copy()
df_out.head(20)


,category_code,category_label,parent_code,text
0,1A2,Unsteady Aerodynamics,1A0,"Unsteady Aerodynamics - In aerodynamics, unste..."
1,1A11,External Noise Prediction,1A0,External Noise Prediction - Prediction of airc...
2,1A6,Wing Design,1A0,Wing Design - The main objective of the wing d...
3,1A7,Aerodynamic of External and Removable Items,1A0,Aerodynamic of External and Removable Items - ...
4,1A9,Wind Tunnel Measuring Techniques,1A0,Wind Tunnel Measuring Techniques - Conventiona...
5,1A1,Computational Fluid Dynamics,1A0,Computational Fluid Dynamics - Computational F...
6,1A10,Computational Acoustics,1A0,Computational Acoustics - Computational method...
7,1A3,Aeronautical Propulsion Integration,1A0,Aeronautical Propulsion Integration - 1. For a...
8,1A4,Airflow Control,1A0,Airflow Control - In this recent and promising...
9,1A5,High Lift Devices,1A0,High Lift Devices - Main objectives of the stu...


Celda para verificar que el parent_code está bien:

In [30]:
# cambiar el code por el que quieras verificar
code = "1A11"
df_out[df_out["category_code"] == code][["category_code", "parent_code", "category_label"]]


,category_code,parent_code,category_label
1,1A11,1A0,External Noise Prediction


En esta celda, podemos verificar si hay alguna categoría sin padre

In [31]:
codes_set = set(df_out["category_code"])
missing_parents = sorted({p for p in df_out["parent_code"].dropna() if p not in codes_set})

print("Parents inexistentes:", missing_parents)


Parents inexistentes: []


Finalmente, guardamos los datos a un fichero CSV utilizando codificación UTF-8 para garantizar compatibilidad con distintos entornos y sistemas.
sformación.

También se descarga automáticamente el fichero generado.

In [32]:
df_out.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"Fichero generado correctamente: {OUTPUT_FILE}")
print(f"Número de filas: {len(df_out)}")

files.download(OUTPUT_FILE)


Fichero generado correctamente: aerotax_transformado.csv
Número de filas: 151


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
INPUT_CSV = "aerotax_transformado.csv"

df_new = pd.read_csv(INPUT_CSV)
df_new.head(20)

,category_code,category_label,parent_code,text
0,1A2,Unsteady Aerodynamics,1A0,"Unsteady Aerodynamics - In aerodynamics, unste..."
1,1A11,External Noise Prediction,1A0,External Noise Prediction - Prediction of airc...
2,1A6,Wing Design,1A0,Wing Design - The main objective of the wing d...
3,1A7,Aerodynamic of External and Removable Items,1A0,Aerodynamic of External and Removable Items - ...
4,1A9,Wind Tunnel Measuring Techniques,1A0,Wind Tunnel Measuring Techniques - Conventiona...
5,1A1,Computational Fluid Dynamics,1A0,Computational Fluid Dynamics - Computational F...
6,1A10,Computational Acoustics,1A0,Computational Acoustics - Computational method...
7,1A3,Aeronautical Propulsion Integration,1A0,Aeronautical Propulsion Integration - 1. For a...
8,1A4,Airflow Control,1A0,Airflow Control - In this recent and promising...
9,1A5,High Lift Devices,1A0,High Lift Devices - Main objectives of the stu...


In [37]:

code = "1D1"
df_new[df_new["category_code"] == code]

,category_code,category_label,parent_code,text
59,1D1,Avionics,1D0,Avionics - DEFINITION: Contains all sub-domain...
